# 018 – Börde Downstream Impact: pandapower Power Flow Validation

**Research question**: Do RMSE differences between disaggregation methods translate into
meaningful differences in downstream power-system decisions?

**Approach**
1. Extract substation-level allocations for six representative methods (RMSE 1.95–10.77 MVA)
2. Build a simplified 110 kV network (13 load buses + 1 slack, MST topology)
3. Run AC power flow for every method *and* for the true load distribution
4. Compare: line thermal loading, voltage deviation, reinforcement trigger errors, cost

> **Note**: Topology is an MST-based approximation; line parameters use German 110 kV
> standard type. Results quantify *relative differences between methods*, not the
> absolute state of the real Avacon grid.

In [ ]:
import sys
import pickle
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandapower as pp
import pandapower.plotting as ppplot
from scipy.spatial.distance import cdist
from scipy.sparse.csgraph import minimum_spanning_tree as mst
from scipy.sparse import csr_matrix
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scienceplots

plt.style.use(['science', 'no-latex'])
warnings.filterwarnings('ignore')

# --- Paths -------------------------------------------------------------------
NOTEBOOK_DIR   = Path('.')
PROJECT_ROOT   = NOTEBOOK_DIR / '../..'
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

INTERMEDIATE_DIR = NOTEBOOK_DIR / 'results' / 'intermediate'
FEATURES_DIR     = INTERMEDIATE_DIR / 'features'
MODELS_DIR       = NOTEBOOK_DIR / 'results' / 'models'
OUTPUT_DIR       = NOTEBOOK_DIR / 'results' / 'pandapower'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Constants ---------------------------------------------------------------
DEMAND_COL      = 'p_mw'
RELATION_COL    = 'Name'
DERIVED_COL     = 'Demand (MVA)'
TARGET_CRS      = 'EPSG:25832'
SEEDS           = [42, 123, 456]
GAMMA_DEFAULT   = 2.0
DIST_CLAMP_KM   = 0.01
RCI_THRESHOLD   = 0.5
PF              = 0.95
TAN_PHI         = float(np.tan(np.arccos(PF)))

# Standard HV/MV transformer ratings (IEC 60076 / German DSO catalogue).
# Each substation is assigned the SMALLEST standard size >= its true peak load.
# This value is INDEPENDENT of TRUE_MW -- a catalogue rating, not a fraction
# of current load.  Using CAP = TRUE_MW * factor makes THRESH = TRUE_MW * k,
# which yields (TRUE_MW > TRUE_MW * k) -- identically False for k > 1.
STD_CAPS_MVA    = np.array([25.0, 40.0, 63.0, 100.0, 160.0])

# N-1 threshold: reinforcement triggered when estimated load > 80% of rated cap.
# Any method that over-estimates a safe substation above this level incurs a
# false-positive reinforcement decision.
N1_THRESHOLD    = 80.0     # % of rated capacity
REINFORCE_COST  = 1.5      # M-EUR per substation (BNetzA reference median)
LINE_RATED_MVA  = np.sqrt(3) * 110 * 0.692   # ~132 MVA per 110 kV line

# --- Display -----------------------------------------------------------------
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 20)

print(f'pandapower {pp.__version__}  |  scienceplots loaded')


## 1  Load Data

In [ ]:
from SpatialAllocation.Allocator import allocator_registry
from SpatialAllocation.Weighter import weighter_registry
from SpatialAllocation.FeatureExtractor.correctors.proximity_corrector import ProximityCorrector

subs_gdf   = gpd.read_file(str(INTERMEDIATE_DIR / 'substations.gpkg'))
region_gdf = gpd.read_file(str(INTERMEDIATE_DIR / 'source_regions.gpkg'))

gemeinde_demand = subs_gdf.groupby('Gemeinde')[DEMAND_COL].sum()
region_gdf[DERIVED_COL] = region_gdf[RELATION_COL].map(gemeinde_demand).fillna(0.0)
TOTAL_MW = subs_gdf[DEMAND_COL].sum()
TRUE_MW  = subs_gdf[DEMAND_COL].values.copy()          # shape (13,)

with open(FEATURES_DIR / 'assembled' / 'boerde_grid_points.pickle', 'rb') as f:
    grid_gdf, _ = pickle.load(f)

ntl_npz    = np.load(str(FEATURES_DIR / 'extracted' / 'boerde_ntl.npz'), allow_pickle=True)
ntl_values = ntl_npz['data'][:, 0]

print(f'{len(subs_gdf)} substations  |  total load {TOTAL_MW:.1f} MW  |  {len(grid_gdf)} grid agents')
print(subs_gdf[['Kennzeichen', DEMAND_COL]].to_string())

## 2  Extract Substation-Level Allocations

In [ ]:
LU_COLS  = ['lu_residential_prop','lu_commercial_prop','lu_industrial_prop',
            'lu_agricultural_prop','lu_others_prop']
PCT_COLS = ['residential_percent','commercial_percent','industrial_percent',
            'agricultural_percent','others_percent']


def compute_demand(grid_df, region_sub, w_result, col='demand'):
    W = w_result.weights
    gdf = grid_df.copy(); gdf[col] = 0.0
    ri = region_sub.set_index(RELATION_COL)
    for name, grp in gdf.groupby(RELATION_COL):
        if name not in ri.index: continue
        total = ri.loc[name, DERIVED_COL]; idx = grp.index
        score = (W[idx] @ np.array([ri.loc[name, c] for c in PCT_COLS])
                 if W.ndim == 2 else W[idx])
        s = score.sum()
        gdf.loc[idx, col] = total * score / s if s > 0 else total / len(grp)
    return gdf


def ntl_correct(grid_df, region_sub, base_col, ntl, out_col):
    """NTL correction: per-Gemeinde epsilon and median (matches production code).

    eps and m_tilde are computed within each Gemeinde loop -- consistent with
    003_boerde_static_allocation.ipynb and ntl_corrector.py (per-ITL3).
    This is the implementation that produces the published RMSE figures.
    """
    rci_mask = ((grid_df['lu_residential_prop'].values
                 + grid_df['lu_commercial_prop'].values
                 + grid_df['lu_industrial_prop'].values) > RCI_THRESHOLD)
    grid_df[out_col] = 0.0
    ri = region_sub.set_index(RELATION_COL)
    for name, grp in grid_df.groupby(RELATION_COL):
        if name not in ri.index: continue
        total = ri.loc[name, DERIVED_COL]; idx = grp.index
        ntl_g = ntl[idx]; rci_g = rci_mask[idx]
        rci_nz = ntl_g[rci_g & (ntl_g > 0)]
        eps = (np.percentile(rci_nz, 5) if len(rci_nz) > 0
               else np.percentile(ntl_g[ntl_g > 0], 5) if (ntl_g > 0).any() else 0.1)
        rci_ntl = ntl_g[rci_g]
        med = np.median(rci_ntl) if len(rci_ntl) > 0 else np.median(ntl_g)
        if med <= 0: med = eps
        fac = np.log(1 + ntl_g + eps) / np.log(1 + med)
        raw = grid_df.loc[idx, base_col].values * fac
        s = raw.sum()
        grid_df.loc[idx, out_col] = total * raw / s if s > 0 else total / len(grp)


def prox_correct(grid_df, region_sub, base_col, prox, out_col):
    """Proximity correction: per-Gemeinde median (matches production code).

    m_tilde_prox is computed within each Gemeinde loop -- consistent with
    proximity_corrector.py (per-ITL3).
    """
    rci_mask = ((grid_df['lu_residential_prop'].values
                 + grid_df['lu_commercial_prop'].values
                 + grid_df['lu_industrial_prop'].values) > RCI_THRESHOLD)
    grid_df[out_col] = 0.0
    ri = region_sub.set_index(RELATION_COL)
    for name, grp in grid_df.groupby(RELATION_COL):
        if name not in ri.index: continue
        total = ri.loc[name, DERIVED_COL]; idx = grp.index
        pg = prox[idx]; rci_p = pg[rci_mask[idx]]
        med = np.median(rci_p) if len(rci_p) > 0 else np.median(pg)
        if med <= 0: med = 1e-6
        fac = np.log(1 + pg) / np.log(1 + med)
        raw = grid_df.loc[idx, base_col].values * fac
        s = raw.sum()
        grid_df.loc[idx, out_col] = total * raw / s if s > 0 else total / len(grp)


def evaluate(true, est):
    rmse = float(np.sqrt(mean_squared_error(true, est)))
    mae  = float(mean_absolute_error(true, est))
    try:  corr, _ = pearsonr(true, est)
    except: corr = float('nan')
    return {'rmse': rmse, 'mae': mae, 'corr': float(corr)}


print('Helper functions defined.')


In [ ]:
# ── Precompute Voronoi assignment (shared across methods) ──
_alloc_v = allocator_registry.create('voronoi')
_vres    = _alloc_v.allocate(grid_gdf, subs_gdf)
V_ASSIGN = _vres.assignment  # (n_grid,)

def agg(demand_arr):
    result = np.zeros(len(subs_gdf))
    for t in range(len(subs_gdf)):
        result[t] = demand_arr[V_ASSIGN == t].sum()
    return result

# ── Uniform base ──
u_w = weighter_registry.create('uniform', config={})
u_r = u_w.compute(grid_gdf, target_gdf=subs_gdf)
grid_gdf = compute_demand(grid_gdf, region_gdf, u_r, col='avg_d')

# ── NTL correction ──
ntl_correct(grid_gdf, region_gdf, 'avg_d', ntl_values, 'ntl_d')

# ── Proximity (gamma=2) correction ──
prox = ProximityCorrector.compute_scores(
    grid_gdf, subs_gdf, gamma=GAMMA_DEFAULT, target_crs=TARGET_CRS, clamp_km=DIST_CLAMP_KM)
prox_correct(grid_gdf, region_gdf, 'ntl_d', prox, 'prox2_ntl_d')

ALL_ALLOC = {
    'true':              TRUE_MW,
    'voronoi_prox2_ntl': agg(grid_gdf['prox2_ntl_d'].values),
    'voronoi_ntl':       agg(grid_gdf['ntl_d'].values),
    'voronoi':           agg(grid_gdf['avg_d'].values),
    'uniform':           np.full(len(subs_gdf), TOTAL_MW / len(subs_gdf)),
}

for k, v in ALL_ALLOC.items():
    m = evaluate(TRUE_MW, v)
    print(f'{k:25s}  sum={v.sum():.1f} MW  RMSE={m["rmse"]:6.3f}')

In [ ]:
# ── GNN allocations (3-seed mean for power flow; per-seed mean RMSE for table) ──
#
# GNN_MAP: (model_config, pickle_key)
#   - voronoi_GNN:          baseline training, raw GNN output -> corresponds to the paper's GNN arm
#   - voronoi_ntl_GNN:      ntl prior training, NTL post-corrected output (intermediate reference method)
#   - voronoi_ntl_prox_GNN: ntl_prox prior training, raw GNN output -> corresponds to the paper's GNNpriorNP arm
#     Note: must use 'gnn_demand' (raw output), not 'ntl_prox_gnn_demand' (post-correction stacked version)
GNN_MAP = {
    'voronoi_GNN':          ('baseline', 'gnn_demand'),
    'voronoi_ntl_GNN':      ('ntl',      'ntl_gnn_demand'),
    'voronoi_ntl_prox_GNN': ('ntl_prox', 'gnn_demand'),   # Fix: raw GNN output, corresponds to GNNpriorNP
}

# GNN_RMSE_MEAN: per-seed mean RMSE for each method, aligned with comparison_table.csv / tab:germany
# Differs from the RMSE of the ensemble-mean allocation in ALL_ALLOC (the latter is lower due to variance reduction)
GNN_RMSE_MEAN = {}

for method, (cfg, key) in GNN_MAP.items():
    arrays = []
    seed_rmses = []
    for seed in SEEDS:
        p = MODELS_DIR / cfg / f'seed_{seed}' / 'grid_demands' / 'boerde_grid_demands.pickle'
        if not p.exists(): continue
        with open(p, 'rb') as f:
            gd = pickle.load(f)
        arr = agg(gd[key])
        arrays.append(arr)
        seed_rmses.append(evaluate(TRUE_MW, arr)['rmse'])
    if arrays:
        mean_a = np.mean(arrays, axis=0)
        ALL_ALLOC[method] = mean_a
        GNN_RMSE_MEAN[method] = float(np.mean(seed_rmses))
        m_ensemble = evaluate(TRUE_MW, mean_a)
        print(f'{method:30s}  sum={mean_a.sum():.1f} MW'
              f'  RMSE(ensemble)={m_ensemble["rmse"]:6.3f}'
              f'  RMSE(per-seed mean)={GNN_RMSE_MEAN[method]:6.3f}')

# Sort by per-seed mean RMSE (consistent ordering with tab:germany)
METHOD_ORDER = ['true'] + sorted(
    [k for k in ALL_ALLOC if k != 'true'],
    key=lambda k: GNN_RMSE_MEAN.get(k, evaluate(TRUE_MW, ALL_ALLOC[k])['rmse'])
)
print('\nMethod order (ascending per-seed mean RMSE):', METHOD_ORDER)

## 3  MST Topology + Network Construction

In [ ]:
# ── Substation coordinates in EPSG:25832 ──
subs_proj = subs_gdf.to_crs(TARGET_CRS)
sub_x = subs_proj.geometry.x.values
sub_y = subs_proj.geometry.y.values

# Slack bus at geographic centroid (represents HV transmission-grid connection)
slack_x, slack_y = sub_x.mean(), sub_y.mean()

# Node matrix: row 0 = slack, rows 1..13 = substations
all_x = np.concatenate([[slack_x], sub_x])
all_y = np.concatenate([[slack_y], sub_y])
all_coords = np.column_stack([all_x, all_y])   # (14, 2)

dist_m = cdist(all_coords, all_coords)
MST    = mst(csr_matrix(dist_m))
ROWS, COLS = MST.nonzero()
EDGE_KM = np.array([dist_m[r, c] / 1000 for r, c in zip(ROWS, COLS)])

print(f'Nodes: 1 slack + 13 substations = {len(all_coords)}')
print(f'MST edges: {len(ROWS)}')
print(f'Line lengths: {EDGE_KM.min():.1f}–{EDGE_KM.max():.1f} km  (mean {EDGE_KM.mean():.1f} km)')

In [ ]:
def build_base_network():
    """Create 14-bus / 13-line pandapower network (no loads)."""
    net = pp.create_empty_network(sn_mva=200, f_hz=50, name='Boerde_110kV_simplified')
    pp.create_std_type(net, {
        'r_ohm_per_km': 0.194,
        'x_ohm_per_km': 0.397,
        'c_nf_per_km':  8.38,
        'max_i_ka':     0.692,
        'type': 'ol',
    }, name='110kV_149AL1', element='line')

    slack_idx = pp.create_bus(net, vn_kv=110.0, name='ExtGrid_Slack',
                              geodata=(float(slack_x), float(slack_y)))

    # Use positional counter (pos) instead of DataFrame label index (i).
    # If subs_gdf was filtered or reset_index upstream, label != position and
    # sub_x[i] would silently produce wrong coordinates without any error.
    sub_idx = [
        pp.create_bus(net, vn_kv=110.0, name=row['Kennzeichen'],
                      geodata=(float(sub_x[pos]), float(sub_y[pos])))
        for pos, (_, row) in enumerate(subs_gdf.iterrows())
    ]
    pp.create_ext_grid(net, bus=slack_idx, vm_pu=1.02, va_degree=0.0, name='TxGrid')

    for r, c in zip(ROWS, COLS):
        fb = slack_idx if r == 0 else sub_idx[r - 1]
        tb = slack_idx if c == 0 else sub_idx[c - 1]
        pp.create_line(net, from_bus=fb, to_bus=tb,
                       length_km=float(dist_m[r, c] / 1000),
                       std_type='110kV_149AL1',
                       name=f'L{r}-{c}')
    return net, slack_idx, sub_idx


BASE_NET, SLACK_BUS, SUB_BUSES = build_base_network()
print(f'Network: {len(BASE_NET.bus)} buses, {len(BASE_NET.line)} lines')
print(f'Line rated capacity: {LINE_RATED_MVA:.0f} MVA/line')


## 4  Network Topology Map (pandapower pplot)

In [ ]:
with plt.style.context(['science', 'no-latex']):

    fig, ax = plt.subplots(figsize=(7, 6))

    # ── Region geometry background ──
    regions_proj = region_gdf.to_crs(TARGET_CRS)
    boerde_proj  = regions_proj.dissolve()  # single Börde boundary

    regions_proj.plot(ax=ax, color='#f4f4f4', edgecolor='#cccccc',
                      linewidth=0.4, zorder=1, label='Gemeinde')
    boerde_proj.boundary.plot(ax=ax, color='#555555', linewidth=1.2, zorder=2)

    # ── pandapower line collection ──
    lc = ppplot.create_line_collection(
        BASE_NET, lines=BASE_NET.line.index,
        color='#2166ac', linewidths=1.8, zorder=3,
        label='110 kV line (MST)')

    # ── pandapower bus collection – load buses ──
    bc = ppplot.create_bus_collection(
        BASE_NET,
        buses=BASE_NET.bus.index[BASE_NET.bus.index != SLACK_BUS],
        size=600, color='#d62728', zorder=4,
        label='Substation (load bus)')

    # ── pandapower bus collection – slack ──
    sc = ppplot.create_bus_collection(
        BASE_NET, buses=[SLACK_BUS],
        size=900, color='#1a9850', zorder=5,
        label='External grid (slack)')

    ppplot.draw_collections([lc, bc, sc], ax=ax)

    # ── Labels: substation name + true load ──
    # for i, row in subs_proj.iterrows():
    #     p_mw = float(subs_gdf.loc[i, DEMAND_COL])
    #     ax.annotate(
    #         f"{row['Kennzeichen']}\n{p_mw:.0f} MW",
    #         xy=(row.geometry.x, row.geometry.y),
    #         xytext=(6, 6), textcoords='offset points',
    #         fontsize=5.5, color='#333333', zorder=6,
    #     )

    # ── Line length labels ──
    # for r, c, lkm in zip(ROWS, COLS, EDGE_KM):
    #     mx = (all_x[r] + all_x[c]) / 2
    #     my = (all_y[r] + all_y[c]) / 2
    #     ax.text(mx, my, f'{lkm:.1f} km', fontsize=4.5, color='#2166ac',
    #             ha='center', va='center', zorder=7)

    ax.set_title('Simplified 110 kV Network (MST approximation)', fontsize=9)
    ax.set_xlabel('Easting EPSG:25832 (m)', fontsize=8)
    ax.set_ylabel('Northing EPSG:25832 (m)', fontsize=8)
    ax.legend(handles=[
        plt.Line2D([0],[0], color='#555555', lw=1.2, label='boundary'),
        plt.Line2D([0],[0], color='#2166ac', lw=1.8, label='110 kV line (MST)'),
        plt.scatter([],[], s=30, c='#d62728', label='Substation (load bus)'),
        plt.scatter([],[], s=45, c='#1a9850', marker='*', label='External grid (slack)'),
    ], fontsize=7, loc='lower right', framealpha=0.8)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'network_topology.pdf'), dpi=300)
    fig.savefig(str(OUTPUT_DIR / 'network_topology.png'), dpi=200)
    plt.show()
    print('Saved: network_topology.pdf / .png')

## 5  Power Flow Simulations (all methods + true baseline)

In [ ]:
def run_pf(method_name, alloc_mw, base_net, sub_buses):
    """Inject loads and run AC power flow (DC fallback). Return result dict."""
    net = copy.deepcopy(base_net)
    for bus, p in zip(sub_buses, alloc_mw):
        pp.create_load(net, bus=bus, p_mw=float(p), q_mvar=float(p)*TAN_PHI)

    mode = 'ac'
    try:
        pp.runpp(net, algorithm='nr', max_iteration=50,
                 calculate_voltage_angles=True, numba=False)
        if not net['converged']: raise RuntimeError('no convergence')
    except Exception:
        try:
            pp.rundcpp(net); mode = 'dc'
        except Exception as e:
            return {'method': method_name, 'converged': False, 'error': str(e)}

    ll  = net.res_line['loading_percent'].values
    vm  = net.res_bus['vm_pu'].values
    vm_sub = vm[1:]   # exclude slack bus

    return {
        'method':       method_name,
        'converged':    True,
        'mode':         mode,
        'alloc_mw':     alloc_mw.tolist(),
        'line_loading': ll.tolist(),
        'bus_vm':       vm.tolist(),
        # aggregate KPIs
        'll_max':       float(ll.max()),
        'll_mean':      float(ll.mean()),
        'll_std':       float(ll.std()),
        'n_over_80':    int((ll > N1_THRESHOLD).sum()),
        'n_over_100':   int((ll > 100).sum()),
        'vm_min':       float(vm_sub.min()),
        'vm_max':       float(vm_sub.max()),
        'n_vm_low':     int((vm_sub < 0.95).sum()),
        'n_vm_high':    int((vm_sub > 1.05).sum()),
    }


# ── Run power flow for every method ──
PF_RESULTS = {}
for name in METHOD_ORDER:
    res = run_pf(name, ALL_ALLOC[name], BASE_NET, SUB_BUSES)
    PF_RESULTS[name] = res
    if res['converged']:
        print(f'[{res["mode"]}] {name:30s}  ll_max={res["ll_max"]:6.1f}%  '
              f'over_80={res["n_over_80"]}  Vmin={res["vm_min"]:.4f}')
    else:
        print(f'[FAIL] {name}  {res.get("error", "")}')

# ── True-case reference arrays ──
TRUE_LL = np.array(PF_RESULTS['true']['line_loading'])   # (13,) line loading %
TRUE_VM = np.array(PF_RESULTS['true']['bus_vm'])         # (14,) voltage pu

## 6  Comparison Against True Power Flow

In [ ]:
# Build per-method delta DataFrame
# RMSE uses the per-seed mean (aligned with tab:germany / comparison_table.csv),
# rather than the RMSE of the ensemble-mean allocation (the two differ due to variance-reduction effects)
delta_rows = []
for name in METHOD_ORDER:
    res = PF_RESULTS[name]
    if not res['converged']: continue
    ll  = np.array(res['line_loading'])
    vm  = np.array(res['bus_vm'])

    rmse_val = GNN_RMSE_MEAN.get(name, evaluate(TRUE_MW, ALL_ALLOC[name])['rmse'])

    delta_ll = ll - TRUE_LL          # per-line delta  (+= more overloaded than true)
    delta_vm = vm[1:] - TRUE_VM[1:]  # per-bus voltage delta

    newly_over_80  = int(((ll > N1_THRESHOLD) & (TRUE_LL <= N1_THRESHOLD)).sum())
    newly_over_100 = int(((ll > 100.0)        & (TRUE_LL <= 100.0)).sum())
    lines_relieved = int(((ll < N1_THRESHOLD) & (TRUE_LL >= N1_THRESHOLD)).sum())

    delta_rows.append({
        'method':             name,
        'RMSE (MVA)':         round(rmse_val, 3),
        'll_max (%)':         round(res['ll_max'], 1),
        'll_std (pp)':        round(res['ll_std'], 2),
        'delta_ll_max (pp)':  round(float(delta_ll.max()), 1),
        'delta_ll_mae (pp)':  round(float(np.mean(np.abs(delta_ll))), 2),
        'delta_ll_rms (pp)':  round(float(np.sqrt((delta_ll**2).mean())), 2),
        'new_over_80':        newly_over_80,
        'new_over_100':       newly_over_100,
        'lines_relieved':     lines_relieved,
        'Vmin (pu)':          round(res['vm_min'], 4),
        'delta_Vmin (pu)':    round(float(delta_vm.min()), 5),
    })

DELTA_DF = pd.DataFrame(delta_rows).set_index('method')
print('=== Power Flow Delta vs True Case ===')
print(DELTA_DF.to_string())
print()
print('Primary discriminating metrics: delta_ll_mae and delta_ll_rms.')
print('new_over_80 / new_over_100 are discrete counts that may be identical')
print('across all methods in a radial conserved-demand network (low info).')

In [ ]:
# Figure: power-flow deviation from true  (3 panels)
#
# WHY delta metrics, not absolute max_loading:
#   All methods conserve total demand.  In this radial MST, accurate methods
#   correctly assign 45 MW to Zielitz (distal node), maximising trunk current.
#   Inaccurate methods smooth the distribution and REDUCE trunk loading despite
#   representing the same total.  This causes absolute max_loading to INCREASE
#   with better RMSE -- opposite to intuition.  The deviation from the TRUE
#   power flow (delta_ll_mae / delta_ll_rms) is the correct fidelity metric.

compare_methods = [m for m in METHOD_ORDER if m != 'true' and PF_RESULTS[m]['converged']]
delta_mae = [DELTA_DF.loc[m, 'delta_ll_mae (pp)']  for m in compare_methods]
delta_max = [DELTA_DF.loc[m, 'delta_ll_max (pp)']  for m in compare_methods]
delta_rms = [DELTA_DF.loc[m, 'delta_ll_rms (pp)']  for m in compare_methods]

short_names = [
    m.replace('voronoi_prox2_ntl', 'Static: Prox+NTL')
     .replace('voronoi_ntl_GNN', 'GNN+NTL')
     .replace('voronoi_GNN', 'GNN')
     .replace('voronoi_ntl_prox_GNN', 'GNN+NTL+Prox')
     .replace('voronoi_ntl', 'Static: NTL')
     .replace('voronoi', 'Static: Voronoi')
     .replace('uniform', 'Uniform')
    for m in compare_methods
]

with plt.style.context(['science', 'no-latex']):
    fig, axes = plt.subplots(1, 3, figsize=(11, 4))
    colors = cm.RdYlGn_r(np.linspace(0.15, 0.85, len(compare_methods)))

    # (a) Per-line loading MAE vs true
    ax = axes[0]
    ax.barh(short_names, delta_mae, color=colors, height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Mean Absolute Loading Deviation vs True (pp)', fontsize=9)
    ax.set_title('(a) Per-Line Loading MAE vs True', fontsize=9)
    ax.tick_params(labelsize=7)

    # (b) Max single-line delta vs true
    ax = axes[1]
    bar_colors = ['#d62728' if d > 0 else '#2ca02c' for d in delta_max]
    ax.barh(short_names, delta_max, color=bar_colors, height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Max Single-Line Loading Delta vs True (pp)', fontsize=9)
    ax.set_title('(b) Worst-Case Line Deviation from True', fontsize=9)
    ax.tick_params(labelsize=7)

    # (c) RMS loading delta vs true
    ax = axes[2]
    ax.barh(short_names, delta_rms, color=colors, height=0.6)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('RMS Loading Deviation vs True (pp)', fontsize=9)
    ax.set_title('(c) Aggregate Loading RMS Error vs True', fontsize=9)
    ax.tick_params(labelsize=7)

    fig.suptitle(
        'Power-Flow Deviation from True Loads -- Borde 110 kV Network\n'
        '(All metrics are relative to the true-load power flow; smaller = better)',
        fontsize=9)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'pf_comparison.pdf'), dpi=300)
    fig.savefig(str(OUTPUT_DIR / 'pf_comparison.png'), dpi=200)
    plt.show()
    print('Saved: pf_comparison.pdf / .png')


In [ ]:
# ── Figure: Per-line delta loading heatmap ──
heatmap_methods = [m for m in METHOD_ORDER if PF_RESULTS[m]['converged']]
line_labels = [f'L{i}' for i in range(len(TRUE_LL))]

delta_matrix = np.array([
    np.array(PF_RESULTS[m]['line_loading']) - TRUE_LL
    for m in heatmap_methods
])  # shape (n_methods, n_lines)

row_labels = [
    m.replace('voronoi_prox2_ntl', 'Static:Prox+NTL')
     .replace('voronoi_ntl_GNN', 'GNN+NTL')
     .replace('voronoi_GNN', 'GNN')
     .replace('voronoi_ntl_prox_GNN', 'GNN+NTL+Prox')
     .replace('voronoi_ntl', 'Static:NTL')
     .replace('voronoi', 'Static:Voronoi')
     .replace('uniform', 'Uniform')
     .replace('true', 'True loads')
    for m in heatmap_methods
]

with plt.style.context(['science', 'no-latex']):
    fig, ax = plt.subplots(figsize=(9, 4))
    vmax = max(abs(delta_matrix).max(), 1.0)
    im = ax.imshow(delta_matrix, aspect='auto',
                   cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Loading delta vs True (pp)', fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    ax.set_xticks(range(len(line_labels)))
    ax.set_xticklabels(line_labels, fontsize=7)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=7)

    # Annotate cells
    for i in range(len(heatmap_methods)):
        for j in range(len(TRUE_LL)):
            val = delta_matrix[i, j]
            ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                    fontsize=5, color='black')

    ax.set_xlabel('Line Index', fontsize=8)
    ax.set_ylabel('Disaggregation Method', fontsize=8)
    ax.set_title('Per-Line Loading Deviation from True Loads (percentage points)', fontsize=9)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'pf_heatmap.pdf'), dpi=300)
    fig.savefig(str(OUTPUT_DIR / 'pf_heatmap.png'), dpi=200)
    plt.show()
    print('Saved: pf_heatmap.pdf / .png')

## 7  Reinforcement Decision Impact Analysis

In [ ]:
# Capacity assignment: smallest IEC standard transformer size >= true peak.
# INDEPENDENT of TRUE_MW.  Previous code used CAP = TRUE_MW * 1.5, making
# THRESH = TRUE_MW * 1.2, so TRUE_TRIG = (TRUE_MW > 1.2 * TRUE_MW) --
# identically False.  FN was always 0 by construction, not observation.

def _std_cap(p_mw):
    above = STD_CAPS_MVA[STD_CAPS_MVA >= p_mw]
    return float(above[0]) if len(above) > 0 else float(STD_CAPS_MVA[-1])

CAP_MW    = np.array([_std_cap(p) for p in TRUE_MW])
THRESH_MW = CAP_MW * (N1_THRESHOLD / 100.0)   # 80% of rated  -> trigger
TRUE_TRIG = TRUE_MW > THRESH_MW                # genuinely at-risk substations

print('=== Substation Capacity Assignment (IEC standard sizes) ===')
df_cap = pd.DataFrame({
    'Kennzeichen': subs_gdf['Kennzeichen'].values,
    'true_MW':     np.round(TRUE_MW, 1),
    'cap_MVA':     CAP_MW,
    'thresh_MW':   np.round(THRESH_MW, 1),
    'util_%':      np.round(TRUE_MW / CAP_MW * 100, 1),
    'at_risk':     TRUE_TRIG,
})
print(df_cap.to_string(index=False))
print(f'\nSubstations genuinely above N-1 threshold ({N1_THRESHOLD:.0f}% of rated): '
      f'{TRUE_TRIG.sum()} / {len(TRUE_TRIG)}')
print('(At-risk substations: missed by a method = False Negative.)')

dec_rows = []
for name in METHOD_ORDER:
    if name == 'true': continue
    est = ALL_ALLOC[name]
    est_trig = est > THRESH_MW
    FP = (~TRUE_TRIG) & est_trig    # false alarm -- unnecessary reinforcement
    FN = TRUE_TRIG  & (~est_trig)   # missed risk -- dangerous substation unplanned
    # Uses the per-seed RMSE mean, aligned with tab:germany / comparison_table.csv
    rmse_val = GNN_RMSE_MEAN.get(name, evaluate(TRUE_MW, est)['rmse'])
    dec_rows.append({
        'method':                name,
        'RMSE (MVA)':            round(rmse_val, 3),
        'FP (false alarm)':      int(FP.sum()),
        'FN (missed risk)':      int(FN.sum()),
        'correct_decisions':     int((est_trig == TRUE_TRIG).sum()),
        'max_abs_err (MVA)':     round(float(np.abs(est - TRUE_MW).max()), 2),
        'unnecessary_cost (ME)': round(int(FP.sum()) * REINFORCE_COST, 2),
    })

DEC_DF = pd.DataFrame(dec_rows).set_index('method').sort_values('RMSE (MVA)')
print()
print('=== Reinforcement Decision Analysis ===')
print(f'Trigger: IEC cap x {N1_THRESHOLD:.0f}%  |  Cost ref: {REINFORCE_COST} M-EUR/sub')
print()
print(DEC_DF.to_string())

In [ ]:
# Figure: RMSE vs downstream impact metrics
plot_methods = [m for m in METHOD_ORDER if m != 'true'
                and m in DEC_DF.index and PF_RESULTS[m]['converged']]

xs = [DEC_DF.loc[m, 'RMSE (MVA)']            for m in plot_methods]
y1 = [DELTA_DF.loc[m, 'delta_ll_mae (pp)']   for m in plot_methods]
y2 = [DELTA_DF.loc[m, 'delta_ll_rms (pp)']   for m in plot_methods]
y3 = [DEC_DF.loc[m, 'FP (false alarm)']       for m in plot_methods]

s_labels = [
    m.replace('voronoi_prox2_ntl', 'Prox+NTL').replace('voronoi_ntl_GNN', 'GNN+NTL')
     .replace('voronoi_GNN', 'GNN').replace('voronoi_ntl_prox_GNN', 'GNN+N+P')
     .replace('voronoi_ntl', 'NTL').replace('voronoi', 'Voronoi')
     .replace('uniform', 'Uniform')
    for m in plot_methods
]

with plt.style.context(['science', 'no-latex']):
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    panels = [
        (axes[0], y1, 'Per-Line Loading MAE vs True (pp)',  '(a) RMSE vs Loading MAE'),
        (axes[1], y2, 'RMS Loading Deviation vs True (pp)',  '(b) RMSE vs Loading RMS'),
        (axes[2], y3, 'False-Alarm Reinforcements (count)',  '(c) RMSE vs FP Decisions'),
    ]
    for ax, ys, ylabel, title in panels:
        sc = ax.scatter(xs, ys, c=xs, cmap='RdYlGn_r', s=60, zorder=3)
        for xi, yi, lab in zip(xs, ys, s_labels):
            ax.annotate(lab, (xi, yi), textcoords='offset points',
                        xytext=(4, 3), fontsize=5.5)
        ax.set_xlabel('Disaggregation RMSE (MVA)', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(title, fontsize=8)
        ax.tick_params(labelsize=7)

    fig.suptitle('Disaggregation Accuracy vs. Downstream System Impact', fontsize=10)
    plt.colorbar(sc, ax=axes[-1], label='RMSE (MVA)', shrink=0.8).ax.tick_params(labelsize=7)
    fig.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'rmse_vs_impact.pdf'), dpi=300)
    fig.savefig(str(OUTPUT_DIR / 'rmse_vs_impact.png'), dpi=200)
    plt.show()
    print('Saved: rmse_vs_impact.pdf / .png')


## 8  Summary Table & Export

In [ ]:
# Merge all result dimensions into one summary table
pf_kpi = pd.DataFrame([
    {'method':      name,
     'll_max (%)':  round(PF_RESULTS[name]['ll_max'], 1),
     'n_over_80':   PF_RESULTS[name]['n_over_80'],
     'Vmin (pu)':   round(PF_RESULTS[name]['vm_min'], 4),
     'pf_mode':     PF_RESULTS[name]['mode'],
    }
    for name in METHOD_ORDER if PF_RESULTS[name]['converged']
]).set_index('method')

delta_sub = DELTA_DF[['delta_ll_max (pp)', 'delta_ll_mae (pp)', 'delta_ll_rms (pp)', 'new_over_80']]

SUMMARY = pf_kpi.join(delta_sub, how='left').join(
    DEC_DF[['RMSE (MVA)', 'FP (false alarm)', 'FN (missed risk)', 'unnecessary_cost (ME)']], how='left')
SUMMARY = SUMMARY.sort_values('RMSE (MVA)')

print('=== Full Summary ===')
print(SUMMARY.to_string())

SUMMARY.to_csv(OUTPUT_DIR / 'downstream_impact_summary.csv')
DELTA_DF.to_csv(OUTPUT_DIR / 'power_flow_delta_vs_true.csv')
DEC_DF.to_csv(OUTPUT_DIR / 'reinforcement_decision_analysis.csv')
print(f'\nResults saved to {OUTPUT_DIR}/')


## 9  Key Findings

In [ ]:
print('=' * 70)
print('KEY FINDINGS  --  Borde 110 kV Downstream Validation')
print('=' * 70)

valid = SUMMARY[SUMMARY.index != 'true'].dropna(subset=['RMSE (MVA)'])
best  = valid.iloc[0]
worst = valid.iloc[-1]

# 1. Accuracy range
print(f"""
Network : 13 substations, 1 slack bus, MST topology
  Total load : {TOTAL_MW:.1f} MW
  Line type  : 110 kV 149-AL1/24-ST1A  (Imax=0.692 kA, ~{LINE_RATED_MVA:.0f} MVA)

1. Disaggregation accuracy
   Best  ({best.name:30s}): RMSE = {best['RMSE (MVA)']:.2f} MVA
   Worst ({worst.name:30s}): RMSE = {worst['RMSE (MVA)']:.2f} MVA
""")

# 2. Power-flow deviation from true
best_mae  = DELTA_DF.loc[best.name,  'delta_ll_mae (pp)']
worst_mae = DELTA_DF.loc[worst.name, 'delta_ll_mae (pp)']
best_rms  = DELTA_DF.loc[best.name,  'delta_ll_rms (pp)']
worst_rms = DELTA_DF.loc[worst.name, 'delta_ll_rms (pp)']
print(f"""2. Power-flow deviation from true loads
   Best  method: loading MAE = {best_mae:.2f} pp  |  RMS = {best_rms:.2f} pp
   Worst method: loading MAE = {worst_mae:.2f} pp  |  RMS = {worst_rms:.2f} pp
   Improvement (worst->best): {worst_mae - best_mae:.2f} pp MAE  /  {worst_rms - best_rms:.2f} pp RMS
""")

# 3. Absolute max-loading inversion (structural artefact -- must be explained)
best_ll  = PF_RESULTS[best.name]['ll_max']
worst_ll = PF_RESULTS[worst.name]['ll_max']
print('3. Absolute max-loading ordering (structural note)')
if best_ll > worst_ll:
    print(f'   Normal direction: lower RMSE -> higher max loading '
          f'({best_ll:.1f}% vs {worst_ll:.1f}%).')
else:
    print(f"""   INVERSION: lower RMSE -> HIGHER max loading
   Best  method max loading : {best_ll:.1f}%
   Worst method max loading : {worst_ll:.1f}%

   Mechanism: all methods conserve total demand ({TOTAL_MW:.0f} MW). In this
   radial MST, accurate methods correctly place 45 MW at Zielitz (a distal
   node), driving high trunk-line current. Inaccurate methods spatially
   smooth demand and reduce trunk current -- not because the grid is safer,
   but because the large load is placed at a closer node.

   => Absolute max_loading is NOT a valid method-comparison metric here.
      Use delta_ll_mae / delta_ll_rms (deviation from the true power flow).""")

# 4. Per-node sensitivity: Zielitz (dominant substation)
print()
print('4. Per-node downstream sensitivity: Zielitz')
zl_mask = subs_gdf['Kennzeichen'].str.contains('Zl', case=False, na=False)
if not zl_mask.any() and 'Name' in subs_gdf.columns:
    zl_mask = subs_gdf['Name'].str.contains('Zielitz', case=False, na=False)
if zl_mask.any():
    zl_label = subs_gdf[zl_mask].index[0]
    zl_pos   = list(subs_gdf.index).index(zl_label)
    true_zl  = TRUE_MW[zl_pos]
    cap_zl   = CAP_MW[zl_pos]
    thr_zl   = THRESH_MW[zl_pos]
    print(f'   True load: {true_zl:.1f} MW  ({100*true_zl/TOTAL_MW:.1f}% of total)')
    print(f'   IEC capacity: {cap_zl:.0f} MVA  |  threshold: {thr_zl:.1f} MW')
    print(f'   {"Method":<30s}  {"Est. MW":>8}  {"Error MW":>9}  {"Util %":>7}')
    print(f'   {"-"*30}  {"-"*8}  {"-"*9}  {"-"*7}')
    for mname in METHOD_ORDER:
        ez = ALL_ALLOC[mname][zl_pos]
        print(f'   {mname:<30s}  {ez:8.1f}  {ez - true_zl:+9.1f}  {100*ez/cap_zl:7.1f}')
else:
    print('   Cannot identify Zielitz -- check Kennzeichen values.')

# 5. Reinforcement decision summary
print()
print(f'5. Reinforcement decisions (IEC cap x {N1_THRESHOLD:.0f}% threshold)')
print(f'   Genuinely at-risk substations: {TRUE_TRIG.sum()} / {len(TRUE_TRIG)}')
if 'FP (false alarm)' in DEC_DF.columns and best.name in DEC_DF.index:
    bfp = int(DEC_DF.loc[best.name,  'FP (false alarm)'])
    wfp = int(DEC_DF.loc[worst.name, 'FP (false alarm)'])
    bfn = int(DEC_DF.loc[best.name,  'FN (missed risk)'])
    wfn = int(DEC_DF.loc[worst.name, 'FN (missed risk)'])
    print(f'   Best  method: FP={bfp}  FN={bfn}  (~{bfp * REINFORCE_COST:.1f} M-EUR unnecessary)')
    print(f'   Worst method: FP={wfp}  FN={wfn}  (~{wfp * REINFORCE_COST:.1f} M-EUR unnecessary)')

print("""
Caveats
  - MST topology approximates the real Avacon grid; absolute values are illustrative
  - Line impedances: std_type 149-AL1; actual Avacon parameters unknown
  - IEC capacities are catalogue minimums, not actual Avacon nameplate ratings
  - All methods conserve total demand; differences arise from spatial redistribution only
  - n = 13; statistical inference not warranted
""")
print('=' * 70)
